In [0]:
# Retail Sales Lakehouse
# Parameterized Pipeline

source_table = "workspace.default.bronze_customers"
target_table = "workspace.default.silver_customers"
load_type = "incremental"

print("Source Table :", source_table)
print("Target Table :", target_table)
print("Load Type    :", load_type)

In [0]:
source_df = spark.table(source_table)

print("Source rows:", source_df.count())

display(source_df)

In [0]:
if load_type.lower() == "full":

    processing_df = source_df

    print("FULL load selected")
    print("Rows selected:", processing_df.count())

elif load_type.lower() == "incremental":

    print("INCREMENTAL load selected")
    print("Source Table :", source_table)
    print("Target Table :", target_table)

else:
    raise ValueError(
        f"Unsupported load_type: {load_type}. "
        "Expected 'full' or 'incremental'."
    )

In [0]:
# Dynamic primary / merge key
merge_key = "customer_id"

print("Source Table :", source_table)
print("Target Table :", target_table)
print("Load Type    :", load_type)
print("Merge Key    :", merge_key)

In [0]:
# Validate source table
if not spark.catalog.tableExists(source_table):
    raise ValueError(f"Source table does not exist: {source_table}")

# Validate target table
if not spark.catalog.tableExists(target_table):
    raise ValueError(f"Target table does not exist: {target_table}")

# Load both tables
source_df = spark.table(source_table)
target_df = spark.table(target_table)

# Validate merge key
if merge_key not in source_df.columns:
    raise ValueError(
        f"Merge key '{merge_key}' not found in source table"
    )

if merge_key not in target_df.columns:
    raise ValueError(
        f"Merge key '{merge_key}' not found in target table"
    )

print("Parameter validation PASSED")
print("Source rows :", source_df.count())
print("Target rows :", target_df.count())
print("Merge key   :", merge_key)

In [0]:
from delta.tables import DeltaTable

def dynamic_delta_merge(source_df, target_table, merge_key):

    target_delta = DeltaTable.forName(
        spark,
        target_table
    )

    merge_condition = (
        f"target.{merge_key} = source.{merge_key}"
    )

    (
        target_delta.alias("target")
        .merge(
            source_df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Dynamic MERGE completed successfully")
    print("Target Table :", target_table)
    print("Merge Key    :", merge_key)

In [0]:
from datetime import date

parameterized_incremental_data = [
    (
        1009,
        "Harry Clark",
        "harry.clark@email.com",
        "Bristol",          # Oxford -> Bristol UPDATE test
        "UK",
        date(2026, 8, 20)
    ),
    (
        1010,
        "Grace Walker",
        "grace.walker@email.com",
        "Glasgow",          # New customer INSERT test
        "UK",
        date(2026, 8, 21)
    )
]

parameterized_incremental_df = spark.createDataFrame(
    parameterized_incremental_data,
    source_df.schema
)

display(parameterized_incremental_df)

In [0]:
dynamic_delta_merge(
    parameterized_incremental_df,
    target_table,
    merge_key
)

In [0]:
from pyspark.sql import functions as F

updated_target_df = spark.table(target_table)

print("Target rows after MERGE:", updated_target_df.count())

display(
    updated_target_df
    .filter(F.col("customer_id").isin(1009, 1010))
    .orderBy("customer_id")
)

In [0]:
dbutils.widgets.text(
    "source_table",
    "workspace.default.bronze_customers",
    "Source Table"
)

dbutils.widgets.text(
    "target_table",
    "workspace.default.silver_customers",
    "Target Table"
)

dbutils.widgets.dropdown(
    "load_type",
    "incremental",
    ["incremental", "full"],
    "Load Type"
)

dbutils.widgets.text(
    "merge_key",
    "customer_id",
    "Merge Key"
)

print("Runtime widgets created successfully")

In [0]:
runtime_source_table = dbutils.widgets.get("source_table")
runtime_target_table = dbutils.widgets.get("target_table")
runtime_load_type = dbutils.widgets.get("load_type")
runtime_merge_key = dbutils.widgets.get("merge_key")

print("Runtime Source Table :", runtime_source_table)
print("Runtime Target Table :", runtime_target_table)
print("Runtime Load Type    :", runtime_load_type)
print("Runtime Merge Key    :", runtime_merge_key)

In [0]:
# Validate load type
valid_load_types = ["full", "incremental"]

if runtime_load_type.lower() not in valid_load_types:
    raise ValueError(
        f"Invalid load_type: {runtime_load_type}"
    )

# Validate source table
if not spark.catalog.tableExists(runtime_source_table):
    raise ValueError(
        f"Source table not found: {runtime_source_table}"
    )

# Validate target table
if not spark.catalog.tableExists(runtime_target_table):
    raise ValueError(
        f"Target table not found: {runtime_target_table}"
    )

# Load runtime tables
runtime_source_df = spark.table(runtime_source_table)
runtime_target_df = spark.table(runtime_target_table)

# Validate merge key
if runtime_merge_key not in runtime_source_df.columns:
    raise ValueError(
        f"Merge key '{runtime_merge_key}' missing in source"
    )

if runtime_merge_key not in runtime_target_df.columns:
    raise ValueError(
        f"Merge key '{runtime_merge_key}' missing in target"
    )

print("Runtime parameter validation PASSED")
print("Load Type   :", runtime_load_type)
print("Source Rows :", runtime_source_df.count())
print("Target Rows :", runtime_target_df.count())
print("Merge Key   :", runtime_merge_key)

In [0]:
if runtime_load_type.lower() == "full":

    print("FULL LOAD mode selected")
    execution_df = runtime_source_df

    print("Rows selected for full load:", execution_df.count())

elif runtime_load_type.lower() == "incremental":

    print("INCREMENTAL LOAD mode selected")

    # For now source batch is prepared separately.
    # We will connect watermark logic in the next step.
    execution_df = runtime_source_df

    print("Source rows available:", execution_df.count())

In [0]:
print("=" * 50)
print("RETAIL SALES LAKEHOUSE - RUNTIME CONFIGURATION")
print("=" * 50)

print(f"Source Table : {runtime_source_table}")
print(f"Target Table : {runtime_target_table}")
print(f"Load Type    : {runtime_load_type}")
print(f"Merge Key    : {runtime_merge_key}")

print("=" * 50)
print("Configuration validation: PASSED")
print("=" * 50)